# Data Cleaning
This is my inital notebook for data cleaning. I am going to use the 2025 data, and I want to re-map all the numeric values from the survey to their true values in the codebook

In [23]:
library(tidyverse)
library(readxl)

npors_2025 <- read_csv('../data/rawdata/NPORS_2025_for_public_release_FINAL.csv')
codebook <- read_excel("../data/rawdata/NPORS_2025_codebook_FINAL.xlsx")


Rows: 5022 Columns: 65
── Column specification ────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
dbl  (63): RESPID, MODE, LANGUAGE, LANGUAGEINITIAL, STRATUM, ECON1MOD, ECON1BMOD, COMTYPE2, UNITY, C...
date  (2): INTERVIEW_START, INTERVIEW_END

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [24]:
#clean codebook to get rid of continuous lables
codebook <- codebook |>
  filter(!str_detect(Value_Labels, "Min|Max"))

In [25]:
#making a lookup table
lookups <- codebook |>
  group_by(Variable) |>
  summarise(map = list(setNames(Value_Labels, Values))) |>
  deframe()

#example
lookups$UNITY


                                                                 1 
 "Americans are united when it comes to the most important values" 
                                                                 2 
"Americans are divided when it comes to the most important values" 
                                                                99 
                                               "Refused/Web blank" 

In [26]:
#applying the labels as factors
npors_2025_labeled <- npors_2025 |>
  mutate(across(
    names(lookups),
    ~ factor(., levels = names(lookups[[cur_column()]]),
             labels = lookups[[cur_column()]])
  ))

In [27]:
# dropping irrelevant columns in the data
npors_2025_labeled <- npors_2025_labeled |> select(-MODE, -LANGUAGE, -LANGUAGEINITIAL, -INTERVIEW_START,
-INTERVIEW_END, -INTFREQ, -RACECMB, -HISP)

In [28]:
head(npors_2025_labeled)

# A tibble: 6 × 57
  RESPID STRATUM       ECON1MOD ECON1BMOD COMTYPE2 UNITY CRIMESAFE GOVPROTCT MOREGUNIMPACT FIN_SIT VET1 
   <dbl> <fct>         <fct>    <fct>     <fct>    <fct> <fct>     <fct>     <fct>         <fct>   <fct>
1   1470 10 = Remaini… Poor     Worse     Rural    Amer… Very safe Sometime… More crime    Just m… No, …
2   2374 7 = 75%+ His… Only fa… Worse     Urban    Amer… Somewhat… It's not… No difference Just m… No, …
3   1177 5 = 50%-74.9… Good     Better    Rural    Amer… Very safe It's not… Less crime    Just m… No, …
4  15459 10 = Remaini… Only fa… About th… Rural    Amer… Very safe Sometime… No difference Don't … No, …
5   9849 9 = Remainin… Good     Better    Suburban Amer… Somewhat… It's not… Less crime    Live c… No, …
6   8178 10 = Remaini… Good     Worse     Urban    Amer… Very safe It's not… Less crime    Live c… No, …
# ℹ 46 more variables: VOL12_CPS <fct>, EMINUSE <fct>, INTMOB <fct>, INTFREQ_COLLAPSED <fct>,
#   HOME4NW2 <fct>, BBHOME <fct>, SMUSE_FB <fct

In [34]:
#now to check for missing values
na_counts <- npors_2025_labeled |>
  summarise(across(everything(), ~sum(is.na(.))))

na_counts <- na_counts |>
  select(where(~.x > 0))

print(na_counts)

# A tibble: 1 × 16
  HOME4NW2 BBHOME SMUSE_FB SMUSE_YT SMUSE_X SMUSE_IG SMUSE_SC SMUSE_WA SMUSE_TT SMUSE_RD SMUSE_BSK
     <int>  <int>    <int>    <int>   <int>    <int>    <int>    <int>    <int>    <int>     <int>
1      176    720      176      176     176      176      176      176      176      176       176
# ℹ 5 more variables: SMUSE_TH <int>, SMUSE_TS <int>, SMART2 <int>, PARTYLN <int>, VOTEGEN_POST <int>


Most of those columns have a small number of missing values, representing 3-15% of the data in that column. However, PARTYLN, which asks for party leaning, is missing 63% of its inputs, and the data (party and leaning) is contained in the variables PARTY and PARTYSUM, so we will remove it.

VOTEGEN_POST is missign 20% of its data, and it asks responders to state who they voted for in the 2020 election.

After removing PARTYLN, we will change all NA values in the dataset to Refused/Web blank

In [ ]:
npors_2025_labeled <- npors_2025_labeled |> select(-PARTYLN)

In [36]:
npors_2025_labeled <- npors_2025_labeled |>
  mutate(across(everything(), ~ replace_na(as.character(.), "Refused/Web blank")))

In [ ]:
na_counts <- npors_2025_labeled |>
  summarise(across(everything(), ~sum(is.na(.))))

na_counts <- na_counts |>
  select(where(~.x > 0))

print(na_counts)
#no na values left

# A tibble: 1 × 0


This dataset is clean, so we will save it, and move on to EDA with it

In [38]:
write_csv(npors_2025_labeled, '../data/cleandata/NPORS_2025_clean.csv')